# 5 - Speed investigation: Rust, conv2d, and batched sources

The forward solve is a Python loop over 1000 timesteps, each launching ~20 tiny tensor ops on a 104x204 grid - so it is **launch/overhead bound**, not FLOP bound. Three ways to attack that overhead, each a clean variant in `fwi.benchmark` (verified equal to the reference `fwi.forward.forward` before timing):

| variant | where | idea |
|---|---|---|
| **rust** | `fwi.rust_solver` (+ `rust/`) | forward solve *and* adjoint gradient in compiled code, wrapped as a `torch.autograd.Function` so it drives L-BFGS |
| **conv2d** | `fwi.forward_conv2d` | the Laplacian as one fused `conv2d` kernel instead of `F.pad` + a Python coefficient loop (autograd- & GPU-friendly) |
| **batched** | `fwi.forward.forward_multishot` | S moving-source shots as one `(S, nI, nJ)` solve, so the GPU's per-launch cost is paid once, not S times |

`fwi.benchmark` exposes them behind one signature (`forward_variants()`, `forward_sequential`/`forward_batched`) plus a device-synced `bench_ms` timer.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/seidlr/acoustic-fwi-ndt-pytorch/blob/main/notebooks/05_speed_investigation.ipynb)

In [ ]:
import sys, os, time
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !git clone -q https://github.com/seidlr/acoustic-fwi-ndt-pytorch.git
    %cd acoustic-fwi-ndt-pytorch
    sys.path.insert(0, os.path.abspath("src"))
    # Build the native Rust solver (~2-3 min: installs rustup + compiles the crate).
    # `pip install ./rust` uses the maturin build backend and needs no virtualenv (Colab
    # has none), unlike `maturin develop`. The notebook still runs without it (cells skip).
    !curl --proto "=https" --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y -q >/dev/null 2>&1
    os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
    !pip -q install ./rust 2>&1 | tail -2

import torch
from fwi.config import resolve_device, resolve_dtype
from fwi.benchmark import bench_ms
from fwi.rust_solver import rust_available

device = resolve_device(); dtype = resolve_dtype(device)
cpu = torch.device("cpu")
print("device:", device, "| dtype:", dtype, "| rust extension:", rust_available())

## 1. Forward variants on CPU (float64)
The Rust solver is CPU/float64; here all variants run there. Rust wins by removing the Python loop + dispatch; conv2d can *lose* on CPU float64 (torch's conv2d is a generic path) - it is a GPU optimization (cell 2).

In [ ]:
from fwi.problems import build_problem
from fwi.benchmark import forward_variants, forward_naive

probc = build_problem('crack', device='cpu', dtype=torch.float64)
ac = (probc.start_alpha2, probc.src_sig, probc.src_i, probc.src_j,
      probc.rec_i, probc.rec_j, probc.cfg)
ref = forward_naive(*ac)
t0 = bench_ms(lambda: forward_naive(*ac), cpu)
print(f'CPU float64 forward  |  naive baseline {t0:6.1f} ms')
for name, fn in forward_variants().items():
    if name == 'naive':
        continue
    rel = float((fn(*ac) - ref).norm() / ref.norm())
    t = bench_ms(lambda fn=fn: fn(*ac), cpu)
    print(f'  {name:7s} {t:6.1f} ms  ({t0/t:.1f}x vs naive)  rel-L2 {rel:.1e}')

## 2. conv2d on the GPU/accelerator
The same fused-conv2d variant on `device`. Run this notebook on a **GPU runtime** to see it win - one kernel launch per Laplacian instead of ~12.

In [ ]:
from fwi.benchmark import forward_conv2d_traces

probd = build_problem('crack', device=device, dtype=dtype)
ad = (probd.start_alpha2, probd.src_sig, probd.src_i, probd.src_j,
      probd.rec_i, probd.rec_j, probd.cfg)
ref_d = forward_naive(*ad)
rel = float((forward_conv2d_traces(*ad) - ref_d).norm() / ref_d.norm())
t_naive = bench_ms(lambda: forward_naive(*ad), device)
t_conv  = bench_ms(lambda: forward_conv2d_traces(*ad), device)
print(f'{device} forward: naive {t_naive:6.1f} ms | conv2d {t_conv:6.1f} ms  '
      f'({t_naive/t_conv:.1f}x)  rel-L2 {rel:.1e}')

## 3. Batched moving sources: S shots in one (S, nI, nJ) solve
`forward_batched` (= `forward_multishot`) vs S sequential single-source solves - identical numbers, one big launch instead of S tiny ones.

In [ ]:
from fwi.benchmark import forward_sequential, forward_batched

ai, aj = probd.rec_i, probd.rec_j           # 16-sensor ring = source positions
sig = probd.src_sig[0] if probd.src_sig.ndim == 2 else probd.src_sig
seq = lambda: forward_sequential(probd.start_alpha2, sig, ai, aj, ai, aj, probd.cfg)
bat = lambda: forward_batched(probd.start_alpha2, sig, ai, aj, ai, aj, probd.cfg)
rel = float((bat() - seq()).norm() / seq().norm())
t_seq = bench_ms(seq, device, k=3)
t_bat = bench_ms(bat, device, k=3)
print(f'{len(ai)}-shot forward ({device}): sequential {t_seq:7.1f} ms | '
      f'batched {t_bat:7.1f} ms  ({t_seq/t_bat:.1f}x)  rel-L2 {rel:.1e}')

## 4. Full L-BFGS inversion: torch-autograd vs the Rust adjoint
`invert_rust` runs the same J0-normalized L-BFGS, but the misfit's forward and gradient are compiled Rust (forward + adjoint-state) instead of torch autograd - so it skips building and walking the autograd tape over 1000 timesteps.

In [ ]:
from fwi.inversion import invert
from fwi.rust_solver import invert_rust

m = probc.active_mask
args = (probc.start_alpha2, probc.observed, probc.src_sig, probc.src_i,
        probc.src_j, probc.rec_i, probc.rec_j, probc.cfg)
t0 = time.perf_counter()
_, hp = invert(*args, active_mask=m, n_iter=12, verbose=False)
tp = time.perf_counter() - t0
print(f'12-iter L-BFGS (CPU f64): torch-autograd {tp:5.2f}s (J/J0 {hp[-1]:.3f})')
if rust_available():
    t0 = time.perf_counter()
    _, hr = invert_rust(*args, active_mask=m, n_iter=12, verbose=False)
    tr = time.perf_counter() - t0
    print(f'                          rust adjoint   {tr:5.2f}s (J/J0 {hr[-1]:.3f})  ->  {tp/tr:.1f}x faster')
else:
    print('Rust extension not built - skipping the Rust-adjoint inversion.')

## Takeaways
- **Rust** wins most on the *inversion*, not the bare forward: it skips the autograd tape (forward + adjoint in one compiled pass). CPU-only here - keeping the GPU would need CUDA/Metal kernels too.
- **conv2d** is the cheapest, stays differentiable, and helps on the **GPU**; on CPU float64 it can be slower - the launch count matters more than the op.
- **Batched sources** is the biggest GPU lever for multi-shot FWI and needs no native code - it just makes each kernel bigger so the device is saturated.
- The diagnosis (overhead-bound, not FLOP-bound) is what makes all three help: they remove launches and Python rather than add compute.